In [ ]:
# Install necessary libraries for Flask API and ngrok tunneling
!pip install flask pyngrok

In [ ]:
# Import core libraries for data handling, model loading, and Flask API
import pandas as pd
import numpy as np
import joblib
from flask import Flask, request, jsonify

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Use Google Colab's file upload utility to get the model file
from google.colab import files

# This command will open a file selection dialog in your browser
uploaded = files.upload()

In [ ]:
# Load the pre-trained machine learning model pipeline
# The model is expected to be named 'alumni_donor_model_pipeline.pkl'
model = joblib.load("alumni_donor_model_pipeline.pkl")

print("Model loaded successfully!")

In [ ]:
# Initialize the Flask application
app = Flask(__name__)

# Define a route for the home endpoint ('/')
@app.route("/")
def home():
    # Return a JSON response indicating the API is running
    return jsonify({
        "message": "Alumni Donor Propensity Forecaster API is running!"
    })

print("Flask application created!")

NameError: name 'Flask' is not defined

In [ ]:
# Re-install necessary libraries (often good practice in a new session or if dependencies are complex)
!pip install flask pyngrok

In [ ]:
# Import core libraries for data manipulation, numerical operations, model handling, and Flask web framework
import pandas as pd
import numpy as np
import joblib

from flask import Flask, request, jsonify

print("✅ Flask and required libraries imported successfully!")

✅ Flask and required libraries imported successfully!


In [ ]:
# Initialize the Flask web application instance
app = Flask(__name__)

# Define the root URL ('/') for the API
@app.route("/")
def home():
    # Return a JSON response for the home endpoint
    return jsonify({
        "message": "Alumni Donor Propensity Forecaster API is running!"
    })

print("✅ Flask application created successfully!")

✅ Flask application created successfully!


In [ ]:
@app.route("/predict", methods=["POST"])
def predict():
    # Ensure model is loaded within the function scope for robustness in Colab
    global model # Declare model as global to ensure it's accessible or loaded.
    try:
        # Attempt to load the model if not already loaded, or re-load if necessary.
        # This makes the function robust to kernel restarts or out-of-order execution.
        if 'model' not in globals():
            model = joblib.load("alumni_donor_model_pipeline.pkl")
    except FileNotFoundError:
        return jsonify({"error": "Model file not found. Please upload 'alumni_donor_model_pipeline.pkl'."}), 500
    except Exception as e:
        return jsonify({"error": f"Failed to load model: {str(e)}"}), 500

    data = request.get_json()

    input_data = pd.DataFrame([{
        "graduation_year": data["graduation_year"],
        "age": data["age"],
        "events_attended": data["events_attended"],
        "emails_received": data["emails_received"],
        "emails_opened": data["emails_opened"],
        "newsletter_clicks": data["newsletter_clicks"],
        "volunteer_events": data["volunteer_events"],
        "previous_donations": data["previous_donations"],
        "total_donation_amount": data["total_donation_amount"],
        "donation_frequency": data["donation_frequency"],
        "days_since_last_donation": data["days_since_last_donation"],
        "career_level": data["career_level"],
        "industry": data["industry"],
        "days_since_last_interaction": data["days_since_last_interaction"]
    }])

    # Feature engineering to create new features for the model
    input_data["years_since_graduation"] = (
        2026 - input_data["graduation_year"]
    )

    input_data["email_open_rate"] = np.where(
        input_data["emails_received"] > 0,
        input_data["emails_opened"] / input_data["emails_received"],
        0
    )

    input_data["email_open_rate"] = input_data["email_open_rate"].clip(0, 1)

    input_data["average_donation_amount"] = np.where(
        input_data["previous_donations"] > 0,
        input_data["total_donation_amount"] /
        input_data["previous_donations"],
        0
    )

    input_data["donation_recency_score"] = (
        1 / (1 + input_data["days_since_last_donation"])
    )

    input_data["event_engagement"] = (
        input_data["events_attended"] / 10
    )

    input_data["newsletter_engagement"] = (
        input_data["newsletter_clicks"] / 15
    )

    input_data["volunteer_engagement"] = (
        input_data["volunteer_events"] / 5
    )

    input_data["interaction_recency"] = (
        1 -
        input_data["days_since_last_interaction"] / 500
    )

    input_data["interaction_recency"] = (
        input_data["interaction_recency"].clip(0, 1)
    )

    input_data["engagement_score"] = (
        0.30 * input_data["event_engagement"]
        + 0.30 * input_data["email_open_rate"]
        + 0.15 * input_data["newsletter_engagement"]
        + 0.15 * input_data["volunteer_engagement"]
        + 0.10 * input_data["interaction_recency"]
    ) * 100

    # Make a prediction using the loaded ML model
    probability = model.predict_proba(input_data)[0][1]

    # Convert probability to a percentage for the propensity score
    propensity_score = probability * 100

    # Categorize the propensity score into High, Medium, or Low
    if propensity_score >= 80:
        category = "High"
        interpretation = "Likely to Donate"

    elif propensity_score >= 50:
        category = "Medium"
        interpretation = "Moderate Donation Likelihood"

    else:
        category = "Low"
        interpretation = "Lower Donation Likelihood"

    # Return the prediction results as a JSON response
    return jsonify({
        "propensity_score": round(propensity_score, 2),
        "category": category,
        "interpretation": interpretation
    })

In [ ]:
# Run the Flask application. This will block the notebook cell.
# For continuous execution without blocking, threading is often used (as shown in later cells).
app.run(host="0.0.0.0", port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


### Set up ngrok Authtoken

To use `pyngrok` for exposing your local Flask app to the internet, you need an authtoken. If you don't have one, you can get it from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). Add your authtoken as a Colab secret named `NGROK_AUTH_TOKEN`. Then, configure `pyngrok` with your token:

In [ ]:
# Import necessary modules for Colab secrets and ngrok client
from google.colab import userdata
from pyngrok import ngrok

# Retrieve the ngrok authentication token from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
# Set the ngrok authtoken for authentication
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✅ ngrok authtoken configured!")

SecretNotFoundError: Secret NGROK_AUTH_TOKEN does not exist.

In [ ]:
# Import the ngrok client
from pyngrok import ngrok

# Connect ngrok to the Flask app running on port 5000
public_url = ngrok.connect(5000)

print("Public API URL:")
print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-09-17T09:21:50+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-09-17T09:21:50+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-09-17T09:21:50+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
# Establish a public URL for the Flask app running on port 5000 using ngrok
public_url = ngrok.connect(5000)

ERROR:pyngrok.process.ngrok:t=2026-09-17T09:25:02+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-09-17T09:25:02+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-09-17T09:25:02+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
# Import threading to run Flask in a separate thread, preventing it from blocking the notebook
import threading

# Define a function to run the Flask application
def run_flask():
    app.run(
        host="0.0.0.0", # Listen on all available public IPs
        port=5000,      # Run on port 5000
        debug=False,    # Disable debug mode for performance/security
        use_reloader=False # Disable reloader to prevent issues with threading
    )

# Create a new thread to run the Flask application
flask_thread = threading.Thread(target=run_flask)
# Start the Flask thread
flask_thread.start()

print("✅ Flask server started")

 * Serving Flask app '__main__'
✅ Flask server started
 * Debug mode: off


In [ ]:
# Import the requests library to make HTTP calls
import requests

# Send a GET request to the local Flask home endpoint
response = requests.get("http://127.0.0.1:5000/")

# Print the HTTP status code and the JSON response from the server
print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 09:27:05] "GET / HTTP/1.1" 200 -


Status Code: 200
Response: {'message': 'Alumni Donor Propensity Forecaster API is running!'}


In [ ]:
# Define the URL for the prediction endpoint of the local Flask API
url = "http://127.0.0.1:5000/predict"

# Print the defined URL
print(url)

http://127.0.0.1:5000/predict


In [ ]:
# Prepare the JSON payload (input data) for the prediction request
payload = {
    "graduation_year": 2020,
    "age": 30,
    "events_attended": 3,
    "emails_received": 20,
    "emails_opened": 10,
    "newsletter_clicks": 5,
    "volunteer_events": 2,
    "previous_donations": 2,
    "total_donation_amount": 1000,
    "donation_frequency": 1,
    "days_since_last_donation": 100,
    "career_level": "Executive",
    "industry": "Finance",
    "days_since_last_interaction": 30
}

# Print the payload to verify its content
print(payload)

{'graduation_year': 2020, 'age': 30, 'events_attended': 3, 'emails_received': 20, 'emails_opened': 10, 'newsletter_clicks': 5, 'volunteer_events': 2, 'previous_donations': 2, 'total_donation_amount': 1000, 'donation_frequency': 1, 'days_since_last_donation': 100, 'career_level': 'Executive', 'industry': 'Finance', 'days_since_last_interaction': 30}


In [ ]:
# Import the requests library to send HTTP requests
import requests

# Send a POST request to the prediction URL with the prepared JSON payload
response = requests.post(
    url,
    json=payload
)

# Print the HTTP status code and the JSON response from the prediction endpoint
print("Status Code:", response.status_code)
print("Response:", response.json())

ERROR:__main__:Exception on /predict [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/tmp/ipykernel_1935/2028705996.py", line 77, in predict
    probability = model.predict_proba(input_data)[0][1]
                  ^^^^^
NameError: name 'model' is not defined
INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 09:47:50] "POST /pred

Status Code: 500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# Import the joblib library for loading the model
import joblib

# Load the machine learning model pipeline from the specified file
model = joblib.load("alumni_donor_model_pipeline.pkl")

print("✅ Model loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'alumni_donor_model_pipeline.pkl'

In [ ]:
# Attempt to load the model using joblib (this cell previously failed due to FileNotFoundError)
joblib.load("alumni_donor_model_pipeline.pkl")

FileNotFoundError: [Errno 2] No such file or directory: 'alumni_donor_model_pipeline.pkl'

In [ ]:
# Import the files utility from google.colab to handle file uploads
from google.colab import files

# This command will open a file selection dialog in your browser to upload a file
uploaded = files.upload()

Saving alumni_donor_model_pipeline.pkl to alumni_donor_model_pipeline.pkl


In [ ]:
# Import the os module to interact with the operating system
import os

# Print a header indicating the list of files
print("Files in Colab:")
# List all files and directories in the current working directory (/content in Colab)
print(os.listdir("/content"))

Files in Colab:
['.config', 'alumni_donor_model_pipeline.pkl', 'sample_data']


In [ ]:
# Import the joblib library for loading serialized Python objects (like ML models)
import joblib

# Define the path to the pre-trained model file
model_path = "/content/alumni_donor_model_pipeline.pkl"

# Load the machine learning model pipeline from the specified path
model = joblib.load(model_path)

print("✅ Model loaded successfully!")
# Print the type of the loaded model to confirm it's a scikit-learn Pipeline
print("Model type:", type(model))

✅ Model loaded successfully!
Model type: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
# Import the files utility from google.colab to handle file downloads
from google.colab import files

# Trigger a download of the specified file to your local machine
files.download("/content/alumni_donor_model_pipeline.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Import Flask components for building the web API
from flask import Flask, request, jsonify

# Initialize the Flask application instance
app = Flask(__name__)

print("✅ Flask application created successfully!")

✅ Flask application created successfully!


In [ ]:
# Define the root endpoint ('/') for the Flask API
@app.route("/")
def home():
    # Return a JSON response for the home page, indicating API status
    return jsonify({
        "message": "Alumni Donor Propensity Forecaster API is running!"
    })

print("✅ Home endpoint created!")

✅ Home endpoint created!


In [ ]:
@app.route("/predict", methods=["POST"])
def predict():
    # Get the JSON data from the request body
    data = request.get_json()

    # Create a Pandas DataFrame from the input data for feature engineering
    input_data = pd.DataFrame([{
        "graduation_year": data["graduation_year"],
        "age": data["age"],
        "events_attended": data["events_attended"],
        "emails_received": data["emails_received"],
        "emails_opened": data["emails_opened"],
        "newsletter_clicks": data["newsletter_clicks"],
        "volunteer_events": data["volunteer_events"],
        "previous_donations": data["previous_donations"],
        "total_donation_amount": data["total_donation_amount"],
        "donation_frequency": data["donation_frequency"],
        "days_since_last_donation": data["days_since_last_donation"],
        "career_level": data["career_level"],
        "industry": data["industry"],
        "days_since_last_interaction": data["days_since_last_interaction"]
    }])

    # Feature engineering steps to transform raw input into features suitable for the model
    input_data["years_since_graduation"] = (
        2026 - input_data["graduation_year"]
    )

    input_data["email_open_rate"] = np.where(
        input_data["emails_received"] > 0,
        input_data["emails_opened"] / input_data["emails_received"],
        0
    )

    input_data["email_open_rate"] = (
        input_data["email_open_rate"].clip(0, 1)
    )

    input_data["average_donation_amount"] = np.where(
        input_data["previous_donations"] > 0,
        input_data["total_donation_amount"] /
        input_data["previous_donations"],
        0
    )

    input_data["donation_recency_score"] = (
        1 / (1 + input_data["days_since_last_donation"])
    )

    input_data["event_engagement"] = (
        input_data["events_attended"] / 10
    )

    input_data["newsletter_engagement"] = (
        input_data["newsletter_clicks"] / 15
    )

    input_data["volunteer_engagement"] = (
        input_data["volunteer_events"] / 5
    )

    input_data["interaction_recency"] = (
        1 -
        input_data["days_since_last_interaction"] / 500
    )

    input_data["interaction_recency"] = (
        input_data["interaction_recency"].clip(0, 1)
    )

    input_data["engagement_score"] = (
        0.30 * input_data["event_engagement"]
        + 0.30 * input_data["email_open_rate"]
        + 0.15 * input_data["newsletter_engagement"]
        + 0.15 * input_data["volunteer_engagement"]
        + 0.10 * input_data["interaction_recency"]
    ) * 100

    # ML prediction using the loaded model
    probability = model.predict_proba(input_data)[0][1]

    # Convert probability to percentage
    propensity_score = probability * 100

    # Categorize propensity into 'High', 'Medium', or 'Low'
    if propensity_score >= 80:
        category = "High"
        interpretation = "Likely to Donate"

    elif propensity_score >= 50:
        category = "Medium"
        interpretation = "Moderate Donation Likelihood"

    else:
        category = "Low"
        interpretation = "Lower Donation Likelihood"

    # Return the prediction and interpretation as a JSON response
    return jsonify({
        "propensity_score": round(propensity_score, 2),
        "category": category,
        "interpretation": interpretation
    })

print("✅ Prediction endpoint created!")

✅ Prediction endpoint created!


In [ ]:
# Import the threading module to run Flask in a separate thread
import threading

# Define the function that will run the Flask app
def run_flask():
    app.run(
        host="0.0.0.0",      # Make the server accessible externally
        port=5000,          # Listen on port 5000
        debug=False,        # Disable debug mode for deployment stability
        use_reloader=False  # Disable reloader when using threading
    )

# Create a new thread for the Flask application
flask_thread = threading.Thread(
    target=run_flask,
    daemon=True             # Make the thread a daemon so it exits with the main program
)

# Start the Flask server in the background thread
flask_thread.start()

print("✅ Flask server started!")

 * Serving Flask app '__main__'
✅ Flask server started!
 * Debug mode: off


Address already in use

In [ ]:
# Import the requests library to send HTTP requests
import requests

# Send a GET request to the local Flask home endpoint
response = requests.get(
    "http://127.0.0.1:5000/"
)

# Print the HTTP status code and the JSON response from the server
print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 09:55:45] "GET / HTTP/1.1" 200 -


Status Code: 200
Response: {'message': 'Alumni Donor Propensity Forecaster API is running!'}


In [ ]:
# Define the URL for the prediction endpoint
url = "http://127.0.0.1:5000/predict"

# Define the JSON payload containing input features for the model
payload = {
    "graduation_year": 2020,
    "age": 30,
    "events_attended": 3,
    "emails_received": 20,
    "emails_opened": 10,
    "newsletter_clicks": 5,
    "volunteer_events": 2,
    "previous_donations": 2,
    "total_donation_amount": 1000,
    "donation_frequency": 1,
    "days_since_last_donation": 100,
    "career_level": "Executive",
    "industry": "Finance",
    "days_since_last_interaction": 30
}

print("JSON input prepared!")

JSON input prepared!


In [ ]:
# Send a POST request to the prediction API with the defined URL and JSON payload
response = requests.post(
    url,
    json=payload
)

# Print the HTTP status code of the response
print("Status Code:", response.status_code)
# Print the raw text content of the response
print("Response:", response.text)

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 09:56:09] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response: {"category":"High","interpretation":"Likely to Donate","propensity_score":82.84}



In [ ]:
# This cell seems to contain a direct JSON output, likely from a previous run or a manual entry.
# It represents a sample prediction result, not executable code.
{
    "category": "High",
    "interpretation": "Likely to Donate",
    "propensity_score": 83.89
}

{'category': 'High',
 'interpretation': 'Likely to Donate',
 'propensity_score': 83.89}

In [ ]:
# Import the threading module to enable running Flask in a separate thread
import threading

# Define a function to encapsulate the Flask app's run method
def run_flask():
    app.run(
        host="0.0.0.0",       # Makes the server accessible externally
        port=5000,           # Specifies the port to run on
        debug=False,         # Disables debug mode for production-like behavior
        use_reloader=False   # Prevents the app from restarting multiple times when running in a thread
    )

# Create a new thread that will execute the `run_flask` function
flask_thread = threading.Thread(
    target=run_flask,
    daemon=True                  # Set as a daemon thread so it terminates when the main program exits
)

# Start the Flask server in the newly created thread
flask_thread.start()

print("✅ Flask server started!")

✅ Flask server started!
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [ ]:
# Define the URL for the prediction endpoint of the Flask API
url = "http://127.0.0.1:5000/predict"

# Print the API endpoint URL for verification
print("API Endpoint:", url)

API Endpoint: http://127.0.0.1:5000/predict


In [ ]:
# Prepare a sample JSON payload (input data) for testing the prediction endpoint
payload = {
    "graduation_year": 2020,
    "age": 30,
    "events_attended": 3,
    "emails_received": 20,
    "emails_opened": 10,
    "newsletter_clicks": 5,
    "volunteer_events": 2,
    "previous_donations": 2,
    "total_donation_amount": 1000,
    "donation_frequency": 1,
    "days_since_last_donation": 100,
    "career_level": "Executive",
    "industry": "Finance",
    "days_since_last_interaction": 30
}

print("✅ JSON input created")
# Print the payload for verification
print(payload)

✅ JSON input created
{'graduation_year': 2020, 'age': 30, 'events_attended': 3, 'emails_received': 20, 'emails_opened': 10, 'newsletter_clicks': 5, 'volunteer_events': 2, 'previous_donations': 2, 'total_donation_amount': 1000, 'donation_frequency': 1, 'days_since_last_donation': 100, 'career_level': 'Executive', 'industry': 'Finance', 'days_since_last_interaction': 30}


In [ ]:
# Import the requests library to send HTTP requests
import requests

# Send a POST request to the prediction endpoint with the defined URL and payload
response = requests.post(
    url,
    json=payload
)

# Print the HTTP status code received from the API
print("Status Code:", response.status_code)

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 09:58:11] "POST /predict HTTP/1.1" 200 -


Status Code: 200


In [ ]:
# Send a POST request to the prediction API using the previously defined URL and payload
response = requests.post(
    url,
    json=payload
)

# Print the HTTP status code of the response
print("Status Code:", response.status_code)
# Print the JSON response from the API
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 09:59:11] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response: {'category': 'High', 'interpretation': 'Likely to Donate', 'propensity_score': 82.84}


In [ ]:
# Parse the JSON response from the API call
result = response.json()

# Print the prediction results in a formatted, easy-to-read manner
print("========== PREDICTION RESULT ==========")
print("Propensity Score :", result["propensity_score"])
print("Category         :", result["category"])
print("Interpretation   :", result["interpretation"])
print("========================================

========== PREDICTION RESULT ==========
Propensity Score : 82.84
Category         : High
Interpretation   : Likely to Donate


In [ ]:
# Print the type of the JSON response to confirm it's a dictionary
print("Response type:", type(response.json()))

Response type: <class 'dict'>


In [ ]:
# Prepare a second JSON payload for another prediction request
payload_2 = {
    "graduation_year": 2015,
    "age": 36,
    "events_attended": 1,
    "emails_received": 30,
    "emails_opened": 3,
    "newsletter_clicks": 1,
    "volunteer_events": 0,
    "previous_donations": 0,
    "total_donation_amount": 0,
    "donation_frequency": 0,
    "days_since_last_donation": 500,
    "career_level": "Mid",
    "industry": "IT",
    "days_since_last_interaction": 400
}

# Send a POST request with the second payload
response_2 = requests.post(
    url,
    json=payload_2
)

# Print the status code and JSON response for the second prediction
print("Status Code:", response_2.status_code)
print("Response:", response_2.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 10:03:20] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response: {'category': 'Medium', 'interpretation': 'Moderate Donation Likelihood', 'propensity_score': 53.55}


In [ ]:
# Prepare a third JSON payload for another prediction request
payload_3 = {
    "graduation_year": 2018,
    "age": 33,
    "events_attended": 8,
    "emails_received": 20,
    "emails_opened": 18,
    "newsletter_clicks": 12,
    "volunteer_events": 5,
    "previous_donations": 5,
    "total_donation_amount": 5000,
    "donation_frequency": 4,
    "days_since_last_donation": 30,
    "career_level": "Executive",
    "industry": "Finance",
    "days_since_last_interaction": 10
}

# Send a POST request with the third payload
response_3 = requests.post(
    url,
    json=payload_3
)

# Print the status code and JSON response for the third prediction
print("Status Code:", response_3.status_code)
print("Response:", response_3.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 10:03:33] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response: {'category': 'High', 'interpretation': 'Likely to Donate', 'propensity_score': 97.49}


In [ ]:
# Send a GET request to a non-existent endpoint to test error handling (404 Not Found)
test_response = requests.get(
    "http://127.0.0.1:5000/abc"
)

# Print the HTTP status code for this test request
print("Status Code:", test_response.status_code)

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 10:03:44] "GET /abc HTTP/1.1" 404 -


Status Code: 404


In [ ]:
# This line is part of a Flask route definition and is incomplete on its own.
# It signifies the start of the /predict endpoint definition.
@app.route("/predict", methods=["POST"])

_IncompleteInputError: incomplete input (3370921315.py, line 1)

In [1]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    return jsonify({
        "message": "Prediction endpoint is working"
    })

In [ ]:
# Conclusion

This notebook demonstrates how to build and deploy a simple Flask API for an alumni donor propensity model, handle file uploads, load the model, and interact with the API endpoints. It also shows how to address common errors related to Flask and ngrok setup.

To ensure all components are running correctly, make sure to:
1. Upload the `alumni_donor_model_pipeline.pkl` file.
2. Configure your `NGROK_AUTH_TOKEN` in Colab secrets.
3. Run all cells sequentially.

This setup provides a local and publicly accessible API for real-time predictions.